# LAB3 — Bài 1: Phân lớp Iris

**Sinh viên:** Trần Chí Nguyên — **MSSV:** 6451071052.

> Mỗi câu lập trình in họ tên và MSSV theo template. Chạy lần lượt từ trên xuống.

In [1]:
%matplotlib inline
HO_TEN = 'Trần Chí Nguyên'
MSSV = '6451071052'
def in_danh_tinh():
    print(f'{HO_TEN} - {MSSV}')


## Câu a–b. Đọc và khảo sát Iris

Nạp dữ liệu từ sklearn, in tên thuộc tính và lớp, kích thước, 5 mẫu đầu và thống kê mô tả.

In [2]:
in_danh_tinh()
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()
X_raw, y = iris.data, iris.target
print('Thuộc tính:', iris.feature_names)
print('Các lớp:', iris.target_names)
print('Số mẫu, số thuộc tính:', X_raw.shape)
print('Shape nhãn:', y.shape)
df = pd.DataFrame(X_raw, columns=iris.feature_names)
df['class'] = pd.Categorical.from_codes(y, iris.target_names)
display(df.head())
display(df.describe(include='all').T)

Trần Chí Nguyên - 6451071052


Thuộc tính: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Các lớp: ['setosa' 'versicolor' 'virginica']
Số mẫu, số thuộc tính: (150, 4)
Shape nhãn: (150,)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),class
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
sepal length (cm),150.0,NaN,NaN,NaN,5.843333,0.828066,4.3,5.1,5.8,6.4,7.9
sepal width (cm),150.0,NaN,NaN,NaN,3.057333,0.435866,2.0,2.8,3.0,3.3,4.4
petal length (cm),150.0,NaN,NaN,NaN,3.758,1.765298,1.0,1.6,4.35,5.1,6.9
petal width (cm),150.0,NaN,NaN,NaN,1.199333,0.762238,0.1,0.3,1.3,1.8,2.5
class,150,3,setosa,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Câu c–d. Chuẩn hóa và chia dữ liệu

Chuẩn hóa từng cột về [0,1] bằng cách chia cho max của cột; chia train/test 8:2.

In [3]:
in_danh_tinh()
from sklearn.model_selection import train_test_split

X = X_raw / X_raw.max(axis=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, y_train.shape)
print('Test:', X_test.shape, y_test.shape)

Trần Chí Nguyên - 6451071052


Train: (120, 4) (120,)
Test: (30, 4) (30,)


## Câu e–h. Huấn luyện và đánh giá bốn mô hình

Decision Tree, Gaussian Naive Bayes, KNN và mạng nơ-ron Dense(10,20,3). In Accuracy, báo cáo Precision/Recall/F1 và confusion matrix heatmap cho từng mô hình.

In [4]:
in_danh_tinh()
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f'\n{name} | Accuracy: {accuracy_score(y_test, pred):.4f}')
    print(classification_report(y_test, pred, target_names=iris.target_names, zero_division=0))
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=iris.target_names, yticklabels=iris.target_names)
    plt.title(f'Confusion matrix — {name}')
    plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.show()

# Keras/TensorFlow theo kiến trúc yêu cầu
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
tf.keras.utils.set_random_seed(42)
models['Neural Network'] = Sequential([
    Dense(10, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(20, activation='relu'),
    Dense(len(iris.target_names), activation='softmax')
])
models['Neural Network'].compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
models['Neural Network'].fit(X_train, y_train, epochs=100, batch_size=1, verbose=1)
proba = models['Neural Network'].predict(X_test, verbose=0)
pred = np.argmax(proba, axis=1)
print('Neural Network | Accuracy:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names, zero_division=0))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title('Confusion matrix — Neural Network'); plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.show()

Trần Chí Nguyên - 6451071052



Decision Tree | Accuracy: 0.9333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30




Naive Bayes | Accuracy: 0.9667
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30




KNN (k=5) | Accuracy: 0.9333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



C:\Users\Nguyen\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100


... (1,205 outputs left out)

114/120 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9561 - loss: 0.0698

120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9583 - loss: 0.0668


Neural Network | Accuracy: 0.9666666666666667
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.91      1.00      0.95        10
   virginica       1.00      0.90      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



  plt.title('Confusion matrix — Neural Network'); plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.show()


## Câu i. Dự đoán ba mẫu mới

Đưa ba mẫu chỉ định vào bốn mô hình đã huấn luyện và in nhãn dự đoán.

In [5]:
in_danh_tinh()
new_samples = np.array([[6.2, 2.9, 4.3, 1.3], [5.1, 3.5, 1.4, 0.2], [7.3, 2.8, 6.4, 2.1]])
new_scaled = new_samples / X_raw.max(axis=0)
for name, model in models.items():
    ids = np.argmax(model.predict(new_scaled, verbose=0), axis=1) if name == 'Neural Network' else model.predict(new_scaled)
    print(name, '=>', iris.target_names[ids])

Trần Chí Nguyên - 6451071052
Decision Tree => ['versicolor' 'setosa' 'virginica']
Naive Bayes => ['versicolor' 'setosa' 'virginica']
KNN (k=5) => ['versicolor' 'setosa' 'virginica']


Neural Network => ['versicolor' 'setosa' 'virginica']


## Câu j. Nhận xét

Đối chiếu Accuracy và báo cáo phân lớp ở trên. Iris có ba lớp và các đặc trưng đo cùng đơn vị; nêu mô hình có kết quả tốt nhất trên tập test, các lớp dễ nhầm và lưu ý mẫu test nhỏ nên kết quả có thể dao động.